### Stage 2: USITC Trade Panel Construction

- Objective: Build the monthly HS4-by-exporter trade panel used as the backbone dataset for next stages.
- Inputs: Raw/cleaned USITC import data.
- Steps: (1) Aggregate to monthly panel level: `country - hs4 - date`; (2) Standardize identifiers (`country` string, `hs4` as 4-digit zero-padded string); (3) Preserve core trade variable(s), especially `import_value`.
- Output: `data/cleaned_data/usitc_imports_hs4_panel.csv`
- This stage serves as the merge base for notification exposure features in stage 3, supports label construction in stage 4 (`Y_struct_abn`), and defines the modeling unit used in analysis stages.

**2.1. Import packages**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

**2.2. Configuration** 

In [ ]:
def _looks_like_root(p: Path) -> bool:
    return (p / 'README.md').exists() and (p / 'code').exists() and (p / 'data').exists()


def _candidate_paths():
    cwd = Path.cwd().resolve()
    seen = set()

    def _push(path: Path):
        p = path.resolve()
        s = str(p)
        if s not in seen:
            seen.add(s)
            return p
        return None

    for p in [cwd, *cwd.parents]:
        q = _push(p)
        if q is not None:
            yield q

    for p in [
        Path('/content/drive/MyDrive/Project'),
        Path('/mnt/g/My Drive/Project'),
        Path(r'G:/My Drive/Project'),
    ]:
        q = _push(p)
        if q is not None:
            yield q


DATA_DIR = next((p for p in _candidate_paths() if _looks_like_root(p)), Path.cwd().resolve())

# New 2010-2025 USITC files (fallback to legacy file if needed)
FILES = [
    DATA_DIR / 'data/raw/USITC 2010-16 SEA.xlsx',
    DATA_DIR / 'data/raw/USITC 2017-22 SEA.xlsx',
    DATA_DIR / 'data/raw/USITC 2023-25 SEA.xlsx',
]
if not all(p.exists() for p in FILES):
    legacy = DATA_DIR / 'data/raw/USITC_data.xlsx'
    FILES = [legacy] if legacy.exists() else FILES

SHEET_CANDIDATES = ['Query Results', 'query results', 'Sheet1']
OUT_DIR = DATA_DIR / 'data/cleaned_data'
OUT_DIR.mkdir(exist_ok=True)
OUT_PANEL_CSV = OUT_DIR / 'usitc_imports_hs4_panel.csv'

print('Using USITC files:')
for f in FILES:
    print('-', f)

Using USITC files:
- G:\My Drive\Project\USITC 2010-16 SEA.xlsx
- G:\My Drive\Project\USITC 2017-22 SEA.xlsx
- G:\My Drive\Project\USITC 2023-25 SEA.xlsx


**2.3. Helper functions**

In [3]:
def parse_number_mixed(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    if s == '':
        return pd.NA
    
    if (',' in s) and ('.' in s):
        if s.rfind(',') > s.rfind('.'):
            s = s.replace('.', '').replace(',', '.')
        else:
            s = s.replace(',', '')
    elif (',' in s) and ('.' not in s):
        s = s.replace(',', '.')
    
    return pd.to_numeric(s, errors='coerce')

def clean_one_file(path: Path) -> pd.DataFrame:
    xls = pd.ExcelFile(path)
    use_sheet = next((s for s in SHEET_CANDIDATES if s in xls.sheet_names), xls.sheet_names[0])
    df = pd.read_excel(path, sheet_name=use_sheet, dtype=str)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    # expect data_type to be in some files but not others, so filter if it exists
    if 'data_type' in df.columns:
        df = df[df['data_type'].str.contains('customs', case=False, na=False)].copy()

    # rename to stable names
    df = df.rename(columns={
        'hts_number': 'hts',
        'general_customs_value': 'import_value',
    })
    # required columns check and type conversion
    needed = ['country', 'year', 'hts', 'import_value', 'month']
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path.name}: {missing}. Found: {df.columns.tolist()}")
    
    # year/month numeric, drop bad rows
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    df['month'] = pd.to_numeric(df['month'], errors='coerce')
    df = df.dropna(subset=['country', 'year', 'month', 'hts', 'import_value']).copy()
    df['year'] = df['year'].astype(int)
    df['month'] = df['month'].astype(int)

    # HS4 from HTS number
    df['hts'] = df['hts'].astype(str)
    df['hs4'] = df['hts'].str.extract(r'^(\d{4})', expand=False)
    df = df.dropna(subset=['hs4']).copy()

    # parse mixed number formats (e.g., 1,234.56 or 1.234,56)
    df['import_value'] = df['import_value'].apply(parse_number_mixed)
    df = df.dropna(subset=['import_value']).copy()
    df['import_value'] = df['import_value'].astype(float)

    # date 
    df['date'] = pd.to_datetime(
        df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2) + '-01',
        errors='coerce'
    )

    # keep description
    if 'description' in df.columns:
        df['description'] = df['description'].astype(str)
    
    # keep only relevant columns
    keep_cols = ['country', 'date', 'hts', 'hs4', 'import_value']
    if 'description' in df.columns:
        keep_cols.append('description')
    return df[keep_cols]

**2.4. Load and concat**

In [4]:
parts = []

# FILES can be a single Path or an iterable of paths
file_list = FILES if isinstance(FILES, (list, tuple, set)) else [FILES]

for f in file_list:
    f = Path(f)
    if not f.exists():
        raise FileNotFoundError(f"File not found: {f}")
    parts.append(clean_one_file(f))

if not parts:
    raise ValueError("No input files were loaded.")

trade = pd.concat(parts, ignore_index=True)

# deduplicate overlaps across split files
trade = trade.sort_values(["country", "hs4", "date"])
trade = trade.drop_duplicates(subset=["country", "hs4", "date"], keep="last")

c:\Users\DGC\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\DGC\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\DGC\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [5]:
print(trade["import_value"].describe())
print("Negative values:", (trade["import_value"] < 0).sum())
print("Missing share:", trade["import_value"].isna().mean())

count    5.257330e+05
mean     5.780212e+06
std      4.286221e+07
min      0.000000e+00
25%      3.585500e+04
50%      2.521170e+05
75%      1.669989e+06
max      3.988419e+09
Name: import_value, dtype: float64
Negative values: 0
Missing share: 0.0


**2.5. Build panel features**

In [6]:
trade["log_import"] = np.log(trade["import_value"] + 1)  # Add 1 to avoid log(0)
trade["growth"] = (
    trade.groupby(["country", "hs4"])["log_import"]
    .diff()
)
trade["growth_3m"] = (
    trade.groupby(["country", "hs4"])["log_import"]
    .diff(3)
) # 3-month growth

# coverage diagnostics
n_countries = trade["country"].nunique()
n_hs4 = trade["hs4"].nunique()
date_min, date_max = trade["date"].min(), trade["date"].max()

print("DONE")
print("row:",  len(trade))
print("countries:", n_countries)
print("HS4 codes:", n_hs4)
print("date range:", date_min, "to", date_max)

# how many months per (country,hs4)?
cov = trade.groupby(["country", "hs4"])["date"].nunique()
print("months per series (p25/median/p75):", cov.quantile([0.25, 0.5, 0.75]).values)

DONE
row: 525733
countries: 8
HS4 codes: 1184
date range: 2010-01-01 00:00:00 to 2025-12-01 00:00:00
months per series (p25/median/p75): [  6.75  53.   176.  ]


In [7]:
# impose minimum length rule
series_len = (
    trade.groupby(["country", "hs4"])["date"]
    .nunique()
    .rename("n_months")
)
trade = trade.merge(series_len, on=["country", "hs4"])
trade = trade[trade["n_months"] >= 24].copy() # minimum total months rule

In [8]:
print("Unique HS4:", trade["hs4"].nunique())
print("Unique exporters:", trade["country"].nunique())
print("Panel size:", len(trade))

print(
    trade.groupby(["hs4","country"])["date"]
      .nunique()
      .describe()
)

Unique HS4: 964
Unique exporters: 8
Panel size: 509310
count    3860.000000
mean      131.945596
std        60.820150
min        24.000000
25%        73.000000
50%       152.000000
75%       192.000000
max       192.000000
Name: date, dtype: float64


In [9]:
## some diagnostics on coverage and missingness
# 1) overall time span
print(trade["date"].min(), trade["date"].max())
print("n months overall:", trade["date"].nunique())

# 2) per exporter coverage
cov_exp = trade.groupby("country")["date"].nunique().sort_values()
print(cov_exp)

# 3) share of missing months per series (relative to full span)
full_T = trade["date"].nunique()
series_T = trade.groupby(["hs4","country"])["date"].nunique()
miss_share = 1 - series_T / full_T
print(miss_share.describe())

2010-01-01 00:00:00 2025-12-01 00:00:00
n months overall: 192
country
Myanmar (Burma)    157
Cambodia           192
Laos               192
Indonesia          192
Malaysia           192
Philippines        192
Thailand           192
Vietnam            192
Name: date, dtype: int64
count    3860.000000
mean        0.312783
std         0.316772
min         0.000000
25%         0.000000
50%         0.208333
75%         0.619792
max         0.875000
Name: date, dtype: float64


**2.6. Save**

In [10]:
trade.to_csv(OUT_PANEL_CSV, index=False, encoding="utf-8")
print("Saved CSV:", OUT_PANEL_CSV)

Saved CSV: G:\My Drive\Project\data/cleaned_data\usitc_imports_hs4_panel.csv
